# ICOS Ocean (SOCAT) — track explorer

Pick a ship/buoy from the first dropdown, then a cruise/deployment from the second.
The track is drawn coloured by **fCO₂** (μatm), masked to the rows where the SOCAT
QC flag is acceptable (≤ 2, the WOCE "good/probably-good" range).

Reads the zarr store via the **datapassport-aware proxy** (`run_proxy.py`).
Default URL is the public proxy at `https://zarr.icos-cp.eu/icos-socat.zarr`;
swap `STORE_URL` to a local instance (e.g. `http://localhost:8001/icos-socat.zarr`)
for offline development.

Map is rendered with **Plotly `Scattergeo`** in the **Natural Earth** projection,
matching the styling used by the FLEXTRA trajectory viewer in `c:\av\python\trajectories`.
Hovering over a point on either subplot highlights the matching point on the other
(`go.FigureWidget` cross-hover, requires `anywidget` on the JupyterLab front-end).

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────
STORE_URL = "https://zarr.icos-cp.eu/icos-socat.zarr"   # public proxy
# STORE_URL = "http://localhost:8001/icos-socat.zarr"   # local proxy fallback

PRIMARY_VAR = "fCO2"           # zarr array name; QC array is f"{PRIMARY_VAR}_QC"
VAR_UNIT    = "μatm"      # μatm
VAR_LABEL   = "fCO₂ in seawater"
QC_KEEP_MAX = 2                # WOCE: 0/1/2 = keep; 3/4/9 = drop

In [2]:
# ── Dependencies ──────────────────────────────────────────────────────────
#!pip install xarray "zarr<3" fsspec aiohttp ipywidgets numpy plotly
import json, urllib.request
import numpy as np
import xarray as xr
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# ── Discover ships and their tracks from .zmetadata ──────────────────────
with urllib.request.urlopen(f"{STORE_URL.rstrip('/')}/.zmetadata", timeout=30) as r:
    meta = json.load(r)
metadata = meta.get("metadata") or {}

cruises = sorted({
    k.split("/", 1)[0]
    for k in metadata
    if k.endswith("/.zgroup") and k.count("/") == 1
    and not k.startswith(".") and not k.startswith("_")
})

ships: dict[str, list[dict]] = {}
for cruise_id in cruises:
    attrs = metadata.get(f"{cruise_id}/.zattrs") or {}
    station = attrs.get("station_id") or attrs.get("platform_name") or "(unknown)"
    ships.setdefault(station, []).append({
        "cruise_id":  cruise_id,
        "time_start": (attrs.get("time_start") or "")[:10],
        "time_end":   (attrs.get("time_end") or "")[:10],
        "fixed":      bool(attrs.get("fixed")),
        "n_rows":     attrs.get("n_rows", "?"),
    })
for k in ships:
    ships[k].sort(key=lambda d: d["time_start"])

print(f"{sum(len(v) for v in ships.values())} cruises across {len(ships)} ship(s)/buoy(s)")

HTTPError: HTTP Error 404: Not Found

In [ ]:
# ── Helpers: load + QC-mask one cruise ────────────────────────────────────
def load_cruise(cruise_id: str) -> dict:
    """Open one cruise group, return QC-masked lon/lat/value/time + percentile bounds."""
    ds = xr.open_zarr(f"{STORE_URL}/{cruise_id}", consolidated=True)
    try:
        if PRIMARY_VAR not in ds:
            raise ValueError(f"{cruise_id}: variable '{PRIMARY_VAR}' not in store")
        if "lon" not in ds or "lat" not in ds:
            raise ValueError(f"{cruise_id}: no lon/lat \u2014 likely a fixed buoy.")

        lon  = ds["lon"].values.astype(float)
        lat  = ds["lat"].values.astype(float)
        val  = ds[PRIMARY_VAR].values.astype(float)
        time = ds["time"].values

        ok = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(val)
        for qc_name in (f"{PRIMARY_VAR}_QC", "lon_QC", "lat_QC"):
            if qc_name in ds:
                qc = ds[qc_name].values
                ok &= (qc >= 0) & (qc <= QC_KEEP_MAX)

        lon, lat, val, time = lon[ok], lat[ok], val[ok], time[ok]
        if val.size == 0:
            return {"lon": np.array([]), "lat": np.array([]), "value": np.array([]),
                    "time": np.array([]), "vmin": None, "vmax": None}

        # Down-sample dense tracks for snappy rendering
        MAX_PTS = 6000
        if val.size > MAX_PTS:
            step = int(np.ceil(val.size / MAX_PTS))
            lon, lat, val, time = lon[::step], lat[::step], val[::step], time[::step]

        vmin = float(np.percentile(val, 2))
        vmax = float(np.percentile(val, 98))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin = float(np.min(val))
            vmax = float(np.max(val)) if float(np.max(val)) > vmin else vmin + 1.0
        return {"lon": lon, "lat": lat, "value": val, "time": time,
                "vmin": vmin, "vmax": vmax}
    finally:
        ds.close()

# 11-stop cmocean.delta sampled at evenly-spaced t — matches the ENVRI app.
# Plotly accepts a colorscale as a list of (t, color) pairs.
CMOCEAN_DELTA_HEX = [
    "#112040", "#163b82", "#345da6", "#6f8fc4", "#a8c0db",
    "#f1eef4",
    "#dcb1a3", "#c97862", "#a64435", "#732420", "#3d0a14",
]
DELTA_PLOTLY = [
    [i / (len(CMOCEAN_DELTA_HEX) - 1), c]
    for i, c in enumerate(CMOCEAN_DELTA_HEX)
]

In [ ]:
# ── Map + timeseries + dropdowns (Plotly Scattergeo, Natural Earth) ───────
# Two-row figure: row 1 = geo (Natural Earth), row 2 = fCO₂ time-series.
# Cross-hover linkage uses go.FigureWidget — Plotly events fire normally
# in the widget channel, so we wire them directly in Python.
#
# Trace indices:
#   0 = geo_highlight   1 = geo_line   2 = geo_points
#   3 = ts_highlight    4 = ts_points  5 = ts_line (broken across gaps)

ship_dd = widgets.Dropdown(
    options=sorted(ships.keys()),
    description="Ship/buoy:",
    layout=widgets.Layout(width="420px"),
)
track_dd = widgets.Dropdown(
    options=[],
    description="Track:",
    layout=widgets.Layout(width="420px"),
)
info = widgets.HTML("<i>Select a ship and track…</i>")

GEO_HL = 0
GEO_PTS = 2
TS_HL = 3
TS_PTS = 4


def _y_with_gaps(times: np.ndarray, values: np.ndarray,
                 gap_factor: float = 4.0) -> list:
    n = len(times)
    if n < 2:
        return list(values)
    diffs = np.diff(times.astype("datetime64[s]").astype("int64"))
    if not diffs.size:
        return list(values)
    median = float(np.median(diffs))
    threshold = max(median * gap_factor, 60.0)
    out: list = [float(values[0])]
    for i in range(1, n):
        if float(diffs[i - 1]) > threshold:
            out.append(None)
        out.append(float(values[i]))
    return out


# Build the FigureWidget once, with empty data, and reuse it across
# every dropdown change.  Updates go through `with fig.batch_update()`
# so the user's pan/zoom (preserved by `uirevision`) survives.
fig = go.FigureWidget(
    make_subplots(
        rows=2, cols=1,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.08,
        specs=[[{"type": "geo"}], [{"type": "xy"}]],
        subplot_titles=("", f"{VAR_LABEL} ({VAR_UNIT})"),
    )
)

# 0 geo highlight
fig.add_trace(go.Scattergeo(
    lon=[], lat=[], mode="markers",
    marker=dict(size=14, color="rgba(255,255,255,0)",
                line=dict(color="#000", width=2)),
    hoverinfo="skip", showlegend=False, name="geo_highlight",
), row=1, col=1)
# 1 geo line
fig.add_trace(go.Scattergeo(
    lon=[], lat=[], mode="lines",
    line=dict(color="#888", width=1),
    opacity=0.45, hoverinfo="skip", showlegend=False, name="geo_line",
), row=1, col=1)
# 2 geo points
fig.add_trace(go.Scattergeo(
    lon=[], lat=[], mode="markers",
    marker=dict(
        size=4, color=[],
        colorscale=DELTA_PLOTLY,
        cmin=0, cmax=1,
        colorbar=dict(title=f"{VAR_LABEL} ({VAR_UNIT})", thickness=12,
                      tickfont=dict(size=10), len=0.6, y=0.78),
    ),
    customdata=[],
    hovertemplate=("%{lat:.3f}°N  %{lon:.3f}°E"
                   "<br>%{marker.color:.1f} " + VAR_UNIT +
                   "<br>%{customdata}<extra></extra>"),
    showlegend=False, name="track",
), row=1, col=1)
# 3 ts highlight
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(size=12, color="rgba(255,255,255,0)",
                line=dict(color="#000", width=2)),
    hoverinfo="skip", showlegend=False, name="ts_highlight",
), row=2, col=1)
# 4 ts points
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(size=4, color=[], colorscale=DELTA_PLOTLY,
                cmin=0, cmax=1, showscale=False),
    customdata=[],
    hovertemplate=("%{x}<br>%{y:.1f} " + VAR_UNIT +
                   "<br>%{customdata[1]:.3f}°N  %{customdata[0]:.3f}°E"
                   "<extra></extra>"),
    showlegend=False, name="ts",
), row=2, col=1)
# 5 ts line (broken at long gaps)
fig.add_trace(go.Scatter(
    x=[], y=[], mode="lines",
    line=dict(color="#888", width=0.6),
    hoverinfo="skip", showlegend=False, name="ts_line",
), row=2, col=1)

fig.update_layout(
    margin=dict(l=0, r=0, t=44, b=40),
    height=820,
    title=dict(text="", x=0.02, xanchor="left", font=dict(size=12)),
    # Drag-to-pan everywhere by default — natural on both the map and
    # the time-series. (Without this Plotly's global default `zoom`
    # makes drags on the *map* trigger a zoom box on the time-series
    # subplot, which is confusing.) Box-zoom is still available via
    # the modebar's zoom button on each subplot.
    dragmode="pan",
    geo=dict(
        projection_type="natural earth",
        resolution=50,
        showland=True,      landcolor="#f5f1e6",
        showocean=True,     oceancolor="#cfe7f5",
        showcoastlines=True, coastlinecolor="#3f3f3f", coastlinewidth=0.8,
        showcountries=True,  countrycolor="#7a7a7a", countrywidth=0.6,
        showsubunits=True,   subunitcolor="#a8a8a8", subunitwidth=0.4,
        showrivers=True,     rivercolor="#8db7d8", riverwidth=0.5,
        showlakes=True,      lakecolor="#cfe7f5",
        bgcolor="white",
    ),
    hovermode="closest",
)
fig.update_xaxes(row=2, col=1, title_text="time", showgrid=True,
                 gridcolor="#eee", showline=True, linecolor="#aaa", ticks="outside")
fig.update_yaxes(row=2, col=1, title_text=f"{VAR_LABEL} ({VAR_UNIT})",
                 showgrid=True, gridcolor="#eee",
                 showline=True, linecolor="#aaa", ticks="outside")


def _on_geo_hover(trace, points, state):
    """Map → timeseries highlight."""
    if not points.point_inds:
        return
    i = points.point_inds[0]
    tsx = list(fig.data[TS_PTS].x or [])
    tsy = list(fig.data[TS_PTS].y or [])
    if i >= len(tsx):
        return
    with fig.batch_update():
        fig.data[TS_HL].x = [tsx[i]]
        fig.data[TS_HL].y = [tsy[i]]


def _on_ts_hover(trace, points, state):
    """Timeseries → map highlight."""
    if not points.point_inds:
        return
    i = points.point_inds[0]
    lon = list(fig.data[GEO_PTS].lon or [])
    lat = list(fig.data[GEO_PTS].lat or [])
    if i >= len(lon):
        return
    with fig.batch_update():
        fig.data[GEO_HL].lon = [lon[i]]
        fig.data[GEO_HL].lat = [lat[i]]


# Bind once — these handlers persist for the life of the FigureWidget.
fig.data[GEO_PTS].on_hover(_on_geo_hover)
fig.data[TS_PTS].on_hover(_on_ts_hover)


def _update_tracks(*_):
    ship = ship_dd.value
    options = [
        (f"{c['cruise_id']}  ({c['time_start']} → {c['time_end']}, n={c['n_rows']})",
         c["cruise_id"])
        for c in ships.get(ship, [])
    ]
    track_dd.options = options
    if options:
        track_dd.value = options[0][1]


def _draw_track(*_):
    cruise_id = track_dd.value
    if not cruise_id:
        return
    info.value = f"<i>Loading <b>{cruise_id}</b> …</i>"
    try:
        d = load_cruise(cruise_id)
    except Exception as exc:
        info.value = f"<span style='color:#b00'>Error: {exc}</span>"
        return
    if not len(d["value"]):
        info.value = f"<span style='color:#b00'>No QC-passing data for {cruise_id}.</span>"
        return

    times_iso = [str(t)[:19] for t in d["time"]]
    south, north = float(d["lat"].min()), float(d["lat"].max())
    west,  east  = float(d["lon"].min()), float(d["lon"].max())
    pad_lon = max(2.0, (east - west) * 0.10)
    pad_lat = max(1.0, (north - south) * 0.10)

    y_line = _y_with_gaps(d["time"], d["value"])
    line_x: list = []
    j = 0
    for v in y_line:
        if v is None:
            line_x.append(None)
        else:
            line_x.append(times_iso[j])
            j += 1

    UIREV = f"socat-{cruise_id}"
    with fig.batch_update():
        # Map line + points
        fig.data[1].lon = list(d["lon"])
        fig.data[1].lat = list(d["lat"])
        fig.data[GEO_PTS].lon = list(d["lon"])
        fig.data[GEO_PTS].lat = list(d["lat"])
        fig.data[GEO_PTS].marker.color = list(d["value"])
        fig.data[GEO_PTS].marker.cmin  = d["vmin"]
        fig.data[GEO_PTS].marker.cmax  = d["vmax"]
        fig.data[GEO_PTS].customdata   = times_iso
        # Reset highlight
        fig.data[GEO_HL].lon = []
        fig.data[GEO_HL].lat = []
        # Timeseries
        fig.data[TS_PTS].x = times_iso
        fig.data[TS_PTS].y = list(d["value"])
        fig.data[TS_PTS].marker.color = list(d["value"])
        fig.data[TS_PTS].marker.cmin  = d["vmin"]
        fig.data[TS_PTS].marker.cmax  = d["vmax"]
        fig.data[TS_PTS].customdata = [(lo, la) for lo, la in zip(d["lon"], d["lat"])]
        # Reset ts highlight
        fig.data[TS_HL].x = []
        fig.data[TS_HL].y = []
        # ts line broken at long gaps
        fig.data[5].x = line_x
        fig.data[5].y = y_line
        # Title + viewport
        fig.layout.title.text = (
            f"{cruise_id} — {VAR_LABEL} "
            f"({d['vmin']:.1f}–{d['vmax']:.1f} {VAR_UNIT}, n={len(d['value'])})"
        )
        # Viewport reset is done via update_layout(geo=...), which accepts
        # the dotted/nested form. Direct attribute access on layout.geo
        # only takes the schema's literal property names.
        fig.update_layout(
            uirevision=UIREV,
            geo=dict(
                uirevision=UIREV,
                lataxis=dict(range=[south - pad_lat, north + pad_lat]),
                lonaxis=dict(range=[west  - pad_lon, east  + pad_lon]),
                center=dict(lat=(south + north) / 2.0, lon=(west + east) / 2.0),
            ),
            xaxis2=dict(uirevision=UIREV),
            yaxis2=dict(uirevision=UIREV),
        )

    info.value = (
        f"<b>{cruise_id}</b> points: {len(d['value'])} "
        f"{VAR_LABEL}: {d['vmin']:.1f} – {d['vmax']:.1f} {VAR_UNIT}"
    )


ship_dd.observe(_update_tracks, names="value")
track_dd.observe(_draw_track,  names="value")
_update_tracks()
_draw_track()

display(widgets.VBox([widgets.HBox([ship_dd, track_dd]), info, fig]))

## Notes

- Colour scale anchored at the **2nd / 98th percentile** of QC-passing fCO₂ for the cruise — same as the ENVRI Data Discovery app, so a few outliers don't blow the gradient.
- Map is **Plotly `Scattergeo` in Natural Earth projection** (the styling used by the FLEXTRA viewer in `c:\av\python\trajectories`). Switch `projection_type` to `"orthographic"` for a globe view, or `"equirectangular"` for a flat plate-carrée.
- Fixed-deployment buoys appear in the dropdown but `load_cruise` raises a friendly error for them (no track to draw). Their time series can be inspected with a separate cell using the same QC mask.
- The ENVRI API `/datasets/{id}/download?format=zarr-track` endpoint returns the same QC-filtered rows as a CSV plus a JSON data passport — no need to re-implement masking in downstream tools.